In [1]:
# =============================================================================
# CHAPTER 4.5 — EXTERNAL VALIDATION: STUFFMART CUSTOMER CONVERSION DATASET
#
# This notebook implements the external robustness check described in
# Chapter 3 Section 3.2.1 of the dissertation.
#
# PURPOSE:
#   Test whether the directional finding from X Education holds in a
#   structurally different conversion context:
#   Does S1 (ML only) outperform S3_uncapped (hard priority without cap)?
#
# SCOPE:
#   Three systems only: S1, S2, S3 (capped and uncapped)
#   No S4 soft adjustment — not implemented for secondary dataset
#   No bootstrap — this is directional robustness checking, not inference
#   Results are compared directionally to X Education primary findings
#
# DATASET:
#   Stuffmart Customer Conversion Dataset (Kaggle)
#   100,000 synthetic leads | ~1.65% conversion rate | Pakistan retail context
#   Two CSVs: training (customer_conversion_traing_dataset.csv)
#             testing  (customer_conversion_testing_dataset.csv)
#
# UPLOAD INSTRUCTIONS:
#   Upload both CSVs to your Google Drive under:
#   /content/drive/MyDrive/Masters_Constrained_Lead_Qualification/
#   Then mount Drive in Cell SM-00.
#
# IMPORTANT — HARDCODED X EDUCATION NUMBERS:
#   Cell SM-10d contains manually entered X Education results for the
#   cross-dataset comparison chart. These are taken from the final
#   primary notebook (Notebook 15). If the primary results change,
#   update XE_PREC and XE_BASELINE in Cell SM-10d accordingly.
#   They are clearly labelled to make this audit trail transparent.
# =============================================================================



In [2]:

# =============================================================================
# CELL SM-00 — SETUP AND MOUNT DRIVE
# =============================================================================

!pip install xgboost -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/Masters_Constrained_Lead_Qualification/'
TRAIN_FILE = DRIVE_PATH + 'customer_conversion_traing_dataset .csv'
TEST_FILE  = DRIVE_PATH + 'customer_conversion_testing_dataset.csv'

print("Setup complete.")



Mounted at /content/drive
Setup complete.


In [3]:

# =============================================================================
# CELL SM-01 — LOAD DATA
# =============================================================================

sm_train_raw = pd.read_csv(TRAIN_FILE)
sm_test_raw  = pd.read_csv(TEST_FILE)

print(f"Training set shape: {sm_train_raw.shape}")
print(f"Test set shape:     {sm_test_raw.shape}")
print(f"\nColumns: {sm_train_raw.columns.tolist()}")
print(f"\nTraining conversion rate: {sm_train_raw['Conversion (Target)'].mean():.4f}")
print(f"Test conversion rate:     {sm_test_raw['Conversion (Target)'].mean():.4f}")
print(f"\nLeadStatus values (train):\n{sm_train_raw['LeadStatus'].value_counts()}")
print(f"\nLeadSource values (train):\n{sm_train_raw['LeadSource'].value_counts()}")
print(f"\nPaymentHistory values (train):\n{sm_train_raw['PaymentHistory'].value_counts()}")



Training set shape: (100000, 19)
Test set shape:     (26145, 19)

Columns: ['LeadID', 'Age', 'Gender', 'Location', 'LeadSource', 'TimeSpent (minutes)', 'PagesViewed', 'LeadStatus', 'EmailSent', 'DeviceType', 'ReferralSource', 'FormSubmissions', 'Downloads', 'CTR_ProductPage', 'ResponseTime (hours)', 'FollowUpEmails', 'SocialMediaEngagement', 'PaymentHistory', 'Conversion (Target)']

Training conversion rate: 0.0165
Test conversion rate:     0.0158

LeadStatus values (train):
LeadStatus
Cold    33435
Hot     33288
Warm    33277
Name: count, dtype: int64

LeadSource values (train):
LeadSource
Organic         25257
Social Media    25030
Email           24947
Referral        24766
Name: count, dtype: int64

PaymentHistory values (train):
PaymentHistory
Good          50111
No Payment    49889
Name: count, dtype: int64


In [4]:
# =============================================================================
# CELL SM-02 — PRE-PROCESSING AND POLICY SIGNAL SETUP
# =============================================================================

TARGET = "Conversion (Target)"

# Final Stuffmart proxy rules
# H1 proxy: commercial risk / low-commitment exclusion
SM_H1_COL   = "PaymentHistory"
SM_H1_VALUE = "No Payment"

# P1 proxy: commercial-attractiveness / human-assessed lead quality
SM_P1_COL   = "LeadStatus"
SM_P1_VALUE = "Hot"

# P2 proxy: inbound/referral channel signal
# NOTE: this is weaker than the primary study's SMS signal because Referral
# may be known at CRM entry. It is retained only as the closest available
# proxy in this secondary directional validation.
SM_P2_COL   = "LeadSource"
SM_P2_VALUE = "Referral"

POLICY_COLS = [SM_H1_COL, SM_P1_COL, SM_P2_COL]

def preprocess_stuffmart(df):
    df = df.copy()
    df = df.drop(columns=["LeadID"], errors="ignore")
    return df

sm_train = preprocess_stuffmart(sm_train_raw)
sm_test  = preprocess_stuffmart(sm_test_raw)

sm_policy_train = sm_train[POLICY_COLS].copy().reset_index(drop=True)
sm_policy_test  = sm_test[POLICY_COLS].copy().reset_index(drop=True)

print("Policy columns preserved:")
print(POLICY_COLS)

print("\nImportant limitation:")
print("LeadSource = Referral is used as the closest available P2 proxy,")
print("but it may be known at CRM entry and is therefore weaker than")
print("the primary study's post-scoring SMS interaction signal.")

Policy columns preserved:
['PaymentHistory', 'LeadStatus', 'LeadSource']

Important limitation:
LeadSource = Referral is used as the closest available P2 proxy,
but it may be known at CRM entry and is therefore weaker than
the primary study's post-scoring SMS interaction signal.


In [5]:

# =============================================================================
# CELL SM-03 — PREPROCESSING PIPELINE
#
# The dataset comes pre-split into two CSVs. We use the provided split.
# This is consistent with the EDA notebook structure submitted alongside
# this study. Stratified splitting is not required here — the files
# already exist. We verify conversion rates are comparable across splits.
# =============================================================================

sm_y_train = sm_train[TARGET]
sm_X_train = sm_train.drop(columns=[TARGET])

sm_y_test  = sm_test[TARGET].reset_index(drop=True)
sm_X_test  = sm_test.drop(columns=[TARGET]).reset_index(drop=True)

print(f"Training: X={sm_X_train.shape}, y={sm_y_train.shape}")
print(f"Test:     X={sm_X_test.shape},  y={sm_y_test.shape}")
print(f"Training conversion rate: {sm_y_train.mean():.4f}")
print(f"Test conversion rate:     {sm_y_test.mean():.4f}")

SM_BASELINE_RATE    = sm_y_test.mean()
SM_TOTAL_CONVERTERS = int(sm_y_test.sum())
SM_K                = int(len(sm_y_test) * 0.2)

print(f"\nBaseline conversion rate : {SM_BASELINE_RATE:.4f}  ({SM_BASELINE_RATE*100:.2f}%)")
print(f"Total converters in test : {SM_TOTAL_CONVERTERS:,}")
print(f"K (top 20%)              : {SM_K:,}")

# Class imbalance: scale_pos_weight = negatives / positives in training set
sm_neg = (sm_y_train == 0).sum()
sm_pos = (sm_y_train == 1).sum()
SM_SCALE_POS_WEIGHT = sm_neg / sm_pos
print(f"\nScale_pos_weight : {SM_SCALE_POS_WEIGHT:.1f}  (negatives/positives in train)")
print(f"  A model predicting all-zero achieves "
      f"{(1 - sm_y_test.mean())*100:.1f}% accuracy — accuracy is uninformative here.")

# Feature types
sm_numeric     = sm_X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
sm_categorical = sm_X_train.select_dtypes(include=['object']).columns.tolist()

print(f"\nNumeric features     ({len(sm_numeric)}): {sm_numeric}")
print(f"Categorical features ({len(sm_categorical)}): {sm_categorical}")

# Preprocessing pipeline — consistent with X Education primary methodology
sm_numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])
sm_cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
sm_preprocessor = ColumnTransformer([
    ('num', sm_numeric_pipe, sm_numeric),
    ('cat', sm_cat_pipe,     sm_categorical),
])


Training: X=(100000, 17), y=(100000,)
Test:     X=(26145, 17),  y=(26145,)
Training conversion rate: 0.0165
Test conversion rate:     0.0158

Baseline conversion rate : 0.0158  (1.58%)
Total converters in test : 413
K (top 20%)              : 5,229

Scale_pos_weight : 59.7  (negatives/positives in train)
  A model predicting all-zero achieves 98.4% accuracy — accuracy is uninformative here.

Numeric features     (10): ['Age', 'TimeSpent (minutes)', 'PagesViewed', 'EmailSent', 'FormSubmissions', 'Downloads', 'CTR_ProductPage', 'ResponseTime (hours)', 'FollowUpEmails', 'SocialMediaEngagement']
Categorical features (7): ['Gender', 'Location', 'LeadSource', 'LeadStatus', 'DeviceType', 'ReferralSource', 'PaymentHistory']


In [ ]:

# =============================================================================
# CELL SM-04 — TRAIN MODELS
#
# Same pipeline structure as X Education primary study.
# Logistic Regression (interpretable baseline) and XGBoost (ensemble).
# Both use class weighting to address severe 1.65% imbalance.
#
# ACCURACY WARNING: The EDA notebook reported 99.5% accuracy for Random
# Forest. This figure is entirely uninformative given the 1.65% baseline
# conversion rate — a model predicting all zeros achieves 98.35% accuracy
# automatically. This notebook evaluates on AUC-ROC and Precision@K only,
# consistent with the primary methodology and appropriate for imbalanced
# classification tasks.
# =============================================================================

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# --- Logistic Regression ---
sm_lr_pipe = Pipeline([
    ('preprocessor', sm_preprocessor),
    ('model', LogisticRegression(
        class_weight='balanced',
        max_iter=2000,
        random_state=RANDOM_STATE,
    )),
])
sm_lr_search = GridSearchCV(
    sm_lr_pipe,
    {'model__C': [0.01, 0.1, 1, 10]},
    scoring='roc_auc', cv=cv, n_jobs=-1, verbose=0,
)
sm_lr_search.fit(sm_X_train, sm_y_train)
sm_lr_auc = roc_auc_score(
    sm_y_test, sm_lr_search.predict_proba(sm_X_test)[:, 1]
)
print(f"Logistic Regression  AUC-ROC: {sm_lr_auc:.4f}  "
      f"(best C: {sm_lr_search.best_params_['model__C']})")

# --- XGBoost ---
sm_xgb_pipe = Pipeline([
    ('preprocessor', sm_preprocessor),
    ('model', XGBClassifier(
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        scale_pos_weight=SM_SCALE_POS_WEIGHT,
    )),
])
sm_xgb_search = RandomizedSearchCV(
    sm_xgb_pipe,
    {
        'model__n_estimators':     [100, 200, 300],
        'model__max_depth':        [2, 3, 4, 5],
        'model__learning_rate':    [0.01, 0.05, 0.1],
        'model__subsample':        [0.7, 0.8, 1.0],
        'model__colsample_bytree': [0.7, 0.8, 1.0],
    },
    n_iter=20, scoring='roc_auc', cv=cv,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=0,
)
sm_xgb_search.fit(sm_X_train, sm_y_train)
sm_xgb_auc = roc_auc_score(
    sm_y_test, sm_xgb_search.predict_proba(sm_X_test)[:, 1]
)
print(f"XGBoost              AUC-ROC: {sm_xgb_auc:.4f}  "
      f"(best params: {sm_xgb_search.best_params_})")

# Select best model by AUC-ROC
if sm_xgb_auc >= sm_lr_auc:
    sm_best_model      = sm_xgb_search
    sm_best_model_name = 'XGBoost'
    sm_best_auc        = sm_xgb_auc
else:
    sm_best_model      = sm_lr_search
    sm_best_model_name = 'Logistic Regression'
    sm_best_auc        = sm_lr_auc

print(f"\nSelected scoring engine: {sm_best_model_name}  (AUC-ROC: {sm_best_auc:.4f})")
print(f"X Education comparison:  XGBoost AUC-ROC 0.8725")

# Generate fixed ML probability vector — consumed unchanged by all systems
sm_ml_scores = sm_best_model.predict_proba(sm_X_test)[:, 1]

print(f"\nML score distribution (test set):")
print(f"  Min:    {sm_ml_scores.min():.4f}")
print(f"  Max:    {sm_ml_scores.max():.4f}")
print(f"  Mean:   {sm_ml_scores.mean():.4f}")
print(f"  Median: {np.median(sm_ml_scores):.4f}")



Logistic Regression  AUC-ROC: 0.9818  (best C: 10)
XGBoost              AUC-ROC: 1.0000  (best params: {'model__subsample': 0.8, 'model__n_estimators': 300, 'model__max_depth': 3, 'model__learning_rate': 0.1, 'model__colsample_bytree': 0.8})

Selected scoring engine: XGBoost  (AUC-ROC: 1.0000)
X Education comparison:  XGBoost AUC-ROC 0.8725

ML score distribution (test set):
  Min:    0.0000
  Max:    0.9993
  Mean:   0.0210
  Median: 0.0000


In [ ]:
# =============================================================================
# CELL SM-04b — LEAKAGE DIAGNOSTIC
#
# XGBoost achieved AUC-ROC = 1.0000. This is not possible on a genuine
# dataset without data leakage. This cell identifies which features are
# causing perfect prediction and decides whether to exclude them.
#
# The most likely culprit is LeadStatus (Cold/Hot/Warm), which the EDA
# notebook showed is strongly associated with conversion. In a synthetic
# dataset, LeadStatus may have been generated directly from the conversion
# label rather than representing a genuine CRM state prior to conversion.
#
# INSERT THIS CELL BETWEEN SM-04 AND SM-05.
# =============================================================================

print("=== LEAKAGE DIAGNOSTIC ===\n")

# Check conversion rate by LeadStatus
print("Conversion rate by LeadStatus (training set):")
ls_check = sm_train_raw.groupby('LeadStatus')['Conversion (Target)'].agg(
    ['mean', 'sum', 'count']
).rename(columns={'mean': 'conv_rate', 'sum': 'converters', 'count': 'total'})
ls_check['conv_rate'] = ls_check['conv_rate'].round(4)
print(ls_check)

# Check conversion rate by Location
print("\nConversion rate by Location (training set):")
loc_check = sm_train_raw.groupby('Location')['Conversion (Target)'].agg(
    ['mean', 'sum', 'count']
).rename(columns={'mean': 'conv_rate', 'sum': 'converters', 'count': 'total'})
loc_check['conv_rate'] = loc_check['conv_rate'].round(4)
print(loc_check.sort_values('conv_rate', ascending=False))

# Check if any numeric feature perfectly separates the classes
print("\nCorrelation of numeric features with target (training set):")
num_corr = sm_train_raw[sm_numeric + ['Conversion (Target)']].corr()[
    'Conversion (Target)'
].drop('Conversion (Target)').abs().sort_values(ascending=False)
print(num_corr.round(4))

# Decision logic
hot_rate = ls_check.loc['Hot', 'conv_rate'] if 'Hot' in ls_check.index else 0
cold_rate = ls_check.loc['Cold', 'conv_rate'] if 'Cold' in ls_check.index else 0

print(f"\nHot lead conversion rate:  {hot_rate:.4f}")
print(f"Cold lead conversion rate: {cold_rate:.4f}")

if hot_rate > 0.9 or (hot_rate > 0.1 and cold_rate < 0.001):
    print("""
  ⚠ LEAKAGE DETECTED: LeadStatus appears to be generated from the
    conversion label in this synthetic dataset. Hot leads convert at
    near-100% and Cold/Warm leads convert at near-0%.

  DECISION: LeadStatus must be EXCLUDED from the ML model feature set.
    It will be retained as a policy-rule-only input (P1 proxy), consistent
    with how Do Not Email/Do Not Call are excluded in the primary study.
    This exclusion is methodologically correct and improves comparability
    with X Education, where policy signals are separated from model features.

  ACTION: Re-run SM-03 and SM-04 after updating MODEL_EXCLUDE below.
""")
    LEAKAGE_CONFIRMED = True
else:
    print("  ✓ No obvious label leakage detected in LeadStatus.")
    LEAKAGE_CONFIRMED = False



=== LEAKAGE DIAGNOSTIC ===

Conversion rate by LeadStatus (training set):
            conv_rate  converters  total
LeadStatus                              
Cold           0.0098         329  33435
Hot            0.0295         982  33288
Warm           0.0101         336  33277

Conversion rate by Location (training set):
            conv_rate  converters  total
Location                                
Islamabad      0.0553         555  10039
Karachi        0.0545         545   9993
Lahore         0.0544         547  10057
Faisalabad     0.0000           0  10196
Gujranwala     0.0000           0  10064
Multan         0.0000           0   9953
Peshawar       0.0000           0  10096
Quetta         0.0000           0   9807
Rawalpindi     0.0000           0   9823
Sialkot        0.0000           0   9972

Correlation of numeric features with target (training set):
PagesViewed              0.1619
EmailSent                0.0653
FollowUpEmails           0.0457
TimeSpent (minutes)      0.

In [ ]:
# =============================================================================
# CELL SM-04c — TRAIN MODELS WITH GOVERNANCE SIGNALS WITHHELD
# =============================================================================

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

# Exclude governance-only signals from ML model
# Location is also excluded because the synthetic data shows strong geographic
# conversion artefacts that are not treated as reliable real-world signal.
SM_MODEL_EXCLUDE = [
    SM_H1_COL,      # PaymentHistory — H1 policy signal
    SM_P1_COL,      # LeadStatus — P1 policy signal / possible synthetic leakage
    SM_P2_COL,      # LeadSource — P2 policy signal
    "Location",     # synthetic geographic artefact
]

sm_X_train = sm_train.drop(columns=[TARGET] + SM_MODEL_EXCLUDE, errors="ignore")
sm_y_train = sm_train[TARGET].astype(int)

sm_X_test = sm_test.drop(columns=[TARGET] + SM_MODEL_EXCLUDE, errors="ignore")
sm_y_test = sm_test[TARGET].astype(int)

numeric_features = sm_X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = sm_X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

lr_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced", solver="liblinear")),
])

lr_grid = {
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100],
}

sm_lr_search = GridSearchCV(
    lr_pipe,
    lr_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
)

sm_lr_search.fit(sm_X_train, sm_y_train)
sm_lr_probs = sm_lr_search.predict_proba(sm_X_test)[:, 1]
sm_lr_auc = roc_auc_score(sm_y_test, sm_lr_probs)

scale_pos_weight = (len(sm_y_train) - sm_y_train.sum()) / max(sm_y_train.sum(), 1)

xgb_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight,
    )),
])

xgb_param_dist = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [2, 3, 4, 5],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__subsample": [0.7, 0.8, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 1.0],
}

sm_xgb_search = RandomizedSearchCV(
    xgb_pipe,
    xgb_param_dist,
    n_iter=20,
    scoring="roc_auc",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

sm_xgb_search.fit(sm_X_train, sm_y_train)
sm_xgb_probs = sm_xgb_search.predict_proba(sm_X_test)[:, 1]
sm_xgb_auc = roc_auc_score(sm_y_test, sm_xgb_probs)

if sm_xgb_auc >= sm_lr_auc:
    sm_best_model = sm_xgb_search
    sm_best_model_name = "XGBoost"
    sm_best_auc = sm_xgb_auc
    sm_ml_scores = sm_xgb_probs
else:
    sm_best_model = sm_lr_search
    sm_best_model_name = "Logistic Regression"
    sm_best_auc = sm_lr_auc
    sm_ml_scores = sm_lr_probs

# Training-only signal map for System 3 robustness blend
sm_signal_train = pd.concat(
    [sm_policy_train[[SM_P1_COL]], sm_y_train.reset_index(drop=True)],
    axis=1
)

SM_LEADSTATUS_SIGNAL_MAP = (
    sm_signal_train
    .groupby(SM_P1_COL)[TARGET]
    .mean()
    .to_dict()
)

SM_DEFAULT_SIGNAL_SCORE = sm_y_train.mean()
SM_POLICY_BASELINE_RATE = sm_y_train.mean()

print("=== STUFFMART MODEL TRAINING WITH GOVERNANCE SIGNALS WITHHELD ===")
print(f"Excluded from ML model: {SM_MODEL_EXCLUDE}")
print(f"Features used by ML model: {sm_X_train.shape[1]}")
print(f"Logistic Regression AUC: {sm_lr_auc:.4f}")
print(f"XGBoost AUC:             {sm_xgb_auc:.4f}")
print(f"Selected scoring engine: {sm_best_model_name} (AUC={sm_best_auc:.4f})")

print("\nTraining-derived LeadStatus signal map for S3:")
print(SM_LEADSTATUS_SIGNAL_MAP)

=== STUFFMART MODEL TRAINING WITH GOVERNANCE SIGNALS WITHHELD ===
Excluded from ML model: ['PaymentHistory', 'LeadStatus', 'LeadSource', 'Location']
Features used by ML model: 13
Logistic Regression AUC: 0.9032
XGBoost AUC:             0.9712
Selected scoring engine: XGBoost (AUC=0.9712)

Training-derived LeadStatus signal map for S3:
{'Cold': 0.009839988036488709, 'Hot': 0.029500120163422253, 'Warm': 0.0100970640382246}


In [ ]:
# =============================================================================
# CELL SM-05 — BUILD RESULTS BASE DATAFRAME
# =============================================================================

sm_results = sm_policy_test.copy()
sm_results["actual"] = sm_y_test.values
sm_results["ml_score"] = sm_ml_scores

SM_BASELINE_RATE = sm_results["actual"].mean()
SM_TOTAL_CONVERTERS = sm_results["actual"].sum()

sm_results["h1_flag"] = sm_results[SM_H1_COL].eq(SM_H1_VALUE)
sm_results["p1_flag"] = sm_results[SM_P1_COL].eq(SM_P1_VALUE)
sm_results["p2_flag"] = sm_results[SM_P2_COL].eq(SM_P2_VALUE)

print("=== STUFFMART RESULTS BASE ===")
print(f"Test set leads:       {len(sm_results):,}")
print(f"Baseline conversion:  {SM_BASELINE_RATE:.4f}")
print(f"Total converters:     {SM_TOTAL_CONVERTERS:,}")

print(f"\nH1 {SM_H1_COL}={SM_H1_VALUE}: {sm_results['h1_flag'].sum():,}")
print(f"P1 {SM_P1_COL}={SM_P1_VALUE}: {sm_results['p1_flag'].sum():,}")
print(f"P2 {SM_P2_COL}={SM_P2_VALUE}: {sm_results['p2_flag'].sum():,}")

print(f"\nPolicy cutoff uses training baseline: {SM_POLICY_BASELINE_RATE:.4f}")

=== STUFFMART RESULTS BASE ===
Test set leads:       26,145
Baseline conversion:  0.0158
Total converters:     413

H1 PaymentHistory=No Payment: 13,123
P1 LeadStatus=Hot: 8,769
P2 LeadSource=Referral: 6,501

Policy cutoff uses training baseline: 0.0165


In [ ]:
# =============================================================================
# CELL SM-06 — APPLY THREE SYSTEMS
# =============================================================================

SM_TIER_CAP_FRACTION = 0.40
SM_BLEND_WEIGHT = 0.90

def apply_sm_systems(
    df,
    tier_cap_fraction=SM_TIER_CAP_FRACTION,
    k_fraction=0.20,
    blend_weight=SM_BLEND_WEIGHT,
    verbose=False,
):
    out = df.copy()

    K = int(len(out) * k_fraction)
    tier_cap = int(K * tier_cap_fraction)

    # H1: operational hard disqualification proxy
    out["hard_disqualified"] = out["h1_flag"]

    contactable = out[~out["hard_disqualified"]].copy()

    # Priority tiers
    contactable["priority_tier"] = 3
    contactable.loc[contactable["p2_flag"], "priority_tier"] = 2
    contactable.loc[contactable["p1_flag"], "priority_tier"] = 1

    # Baseline cutoff: only elevate if ML score also supports the lead
    below_cutoff = contactable["ml_score"] < SM_POLICY_BASELINE_RATE
    contactable.loc[
        below_cutoff & contactable["priority_tier"].lt(3),
        "priority_tier"
    ] = 3

    def build_tiercapped_ranking(eligible_df):
        p1 = eligible_df[eligible_df["priority_tier"] == 1].sort_values("ml_score", ascending=False)
        p2 = eligible_df[eligible_df["priority_tier"] == 2].sort_values("ml_score", ascending=False)
        p3 = eligible_df[eligible_df["priority_tier"] == 3].sort_values("ml_score", ascending=False)

        selected_p1 = p1.head(tier_cap)
        selected_p2 = p2.head(tier_cap)

        overflow = pd.concat(
            [p1.iloc[tier_cap:], p2.iloc[tier_cap:], p3],
            ignore_index=True
        ).sort_values("ml_score", ascending=False)

        return pd.concat([selected_p1, selected_p2, overflow], ignore_index=True)

    # System 3 signal blend
    contactable["signal_score"] = (
        contactable[SM_P1_COL]
        .map(SM_LEADSTATUS_SIGNAL_MAP)
        .fillna(SM_DEFAULT_SIGNAL_SCORE)
    )

    contactable["blended_score"] = (
        blend_weight * contactable["ml_score"]
        + (1 - blend_weight) * contactable["signal_score"]
    )

    if verbose:
        print("=== STUFFMART SYSTEM DIAGNOSTICS ===")
        print(f"K={K:,}")
        print(f"Tier cap={tier_cap:,} ({tier_cap_fraction:.0%} of K)")
        print(f"H1 disqualified: {out['hard_disqualified'].sum():,}")
        print(f"Contactable pool: {len(contactable):,}")
        print("\nPriority tiers after baseline cutoff:")
        print(contactable["priority_tier"].value_counts().sort_index())
        print("\nNote: Stuffmart is directional validation only, not replication.")

    return {
        "System 1 — ML baseline":
            contactable.sort_values("ml_score", ascending=False),

        "System 2 — Tier qualification":
            build_tiercapped_ranking(contactable),

        "System 3 — Calibrated blend":
            contactable.sort_values("blended_score", ascending=False),
    }

sm_systems = apply_sm_systems(sm_results, verbose=True)
print("\nStuffmart systems applied.")

=== STUFFMART SYSTEM DIAGNOSTICS ===
K=5,229
Tier cap=2,091 (40% of K)
H1 disqualified: 13,123
Contactable pool: 13,022

Priority tiers after baseline cutoff:
priority_tier
1      919
2      433
3    11670
Name: count, dtype: int64

Note: Stuffmart is directional validation only, not replication.

Stuffmart systems applied.


In [ ]:
# =============================================================================
# CELL SM-07 — EVALUATE SYSTEMS
# =============================================================================

def evaluate_sm_systems(systems, k_fraction=0.20):
    K = int(len(sm_results) * k_fraction)

    rows = []
    for name, ranked_df in systems.items():
        selected = ranked_df.head(K)

        precision = selected["actual"].mean()
        recall = selected["actual"].sum() / SM_TOTAL_CONVERTERS
        lift = precision / SM_BASELINE_RATE
        wasted = 1 - precision

        rows.append({
            "system": name,
            "K": K,
            "eligible_leads": len(ranked_df),
            "leads_selected": len(selected),
            "converters_in_K": int(selected["actual"].sum()),
            "precision_at_k": round(precision, 6),
            "recall_at_k": round(recall, 6),
            "lift_at_k": round(lift, 6),
            "wasted_effort_rate": round(wasted, 6),
        })

    return pd.DataFrame(rows)

sm_main_results = evaluate_sm_systems(sm_systems, k_fraction=0.20)

print("=== STUFFMART VALIDATION — SYSTEM COMPARISON (K=20%) ===")
print(sm_main_results.to_string(index=False))

sm_main_results.to_csv("table_SM_main_results.csv", index=False)
print("\nExported: table_SM_main_results.csv")

=== STUFFMART VALIDATION — SYSTEM COMPARISON (K=20%) ===
                       system    K  eligible_leads  leads_selected  converters_in_K  precision_at_k  recall_at_k  lift_at_k  wasted_effort_rate
       System 1 — ML baseline 5229           13022            5229              197        0.037675     0.476998   2.384988            0.962325
System 2 — Tier qualification 5229           13022            5229              197        0.037675     0.476998   2.384988            0.962325
  System 3 — Calibrated blend 5229           13022            5229              197        0.037675     0.476998   2.384988            0.962325

Exported: table_SM_main_results.csv


In [ ]:
# =============================================================================
# CELL SM-08 — TIER CAP SENSITIVITY
# =============================================================================

sm_cap_rows = []

for cap in [0.20, 0.40, 0.60, 1.00]:
    sys_run = apply_sm_systems(
        sm_results,
        tier_cap_fraction=cap,
        k_fraction=0.20,
        verbose=False,
    )

    eval_df = evaluate_sm_systems(sys_run, k_fraction=0.20)
    eval_df["tier_cap_fraction"] = cap
    sm_cap_rows.append(eval_df)

sm_cap_results = pd.concat(sm_cap_rows, ignore_index=True)

sm_cap_pivot = sm_cap_results.pivot_table(
    index="tier_cap_fraction",
    columns="system",
    values="precision_at_k",
).round(4)

print("=== STUFFMART TIER CAP SENSITIVITY — Precision@K ===")
print(sm_cap_pivot.to_string())

sm_cap_results.to_csv("table_SM_cap_sensitivity.csv", index=False)
print("\nExported: table_SM_cap_sensitivity.csv")

=== STUFFMART TIER CAP SENSITIVITY — Precision@K ===
system             System 1 — ML baseline  System 2 — Tier qualification  System 3 — Calibrated blend
tier_cap_fraction                                                                                    
0.2                                0.0377                         0.0377                       0.0377
0.4                                0.0377                         0.0377                       0.0377
0.6                                0.0377                         0.0377                       0.0377
1.0                                0.0377                         0.0377                       0.0377

Exported: table_SM_cap_sensitivity.csv


In [ ]:
# =============================================================================
# CELL SM-09 — K SENSITIVITY
# =============================================================================

sm_k_rows = []

for k_frac in [0.10, 0.20, 0.30]:
    sys_run = apply_sm_systems(
        sm_results,
        tier_cap_fraction=SM_TIER_CAP_FRACTION,
        k_fraction=k_frac,
        verbose=False,
    )

    eval_df = evaluate_sm_systems(sys_run, k_fraction=k_frac)
    eval_df["k_fraction"] = k_frac
    sm_k_rows.append(eval_df)

sm_k_results = pd.concat(sm_k_rows, ignore_index=True)

sm_k_pivot = sm_k_results.pivot_table(
    index="k_fraction",
    columns="system",
    values="precision_at_k",
).round(4)

print("=== STUFFMART K SENSITIVITY — Precision@K ===")
print(sm_k_pivot.to_string())

sm_k_results.to_csv("table_SM_k_sensitivity.csv", index=False)
print("\nExported: table_SM_k_sensitivity.csv")

=== STUFFMART K SENSITIVITY — Precision@K ===
system      System 1 — ML baseline  System 2 — Tier qualification  System 3 — Calibrated blend
k_fraction                                                                                    
0.1                         0.0754                         0.0754                       0.0754
0.2                         0.0377                         0.0377                       0.0377
0.3                         0.0251                         0.0251                       0.0251

Exported: table_SM_k_sensitivity.csv


In [ ]:
# =============================================================================
# CELL SM-10 — VISUALISATIONS (FIXED)
# Updated system names to match final three-system architecture.
# =============================================================================

systems_order = [
    'System 1 — ML only',
    'System 2 — Hard governance',
    'System 3 — Calibrated blend',
]
colors = ['#90CAF9', '#1E88E5', '#42A5F5']

s1 = sm_main_results[sm_main_results['system']=='System 1 — ML only']['precision_at_k'].values[0]
s2 = sm_main_results[sm_main_results['system']=='System 2 — Hard governance']['precision_at_k'].values[0]
s3 = sm_main_results[sm_main_results['system']=='System 3 — Calibrated blend']['precision_at_k'].values[0]
l1 = sm_main_results[sm_main_results['system']=='System 1 — ML only']['lift_at_k'].values[0]
l2 = sm_main_results[sm_main_results['system']=='System 2 — Hard governance']['lift_at_k'].values[0]
l3 = sm_main_results[sm_main_results['system']=='System 3 — Calibrated blend']['lift_at_k'].values[0]

# --- SM-10a: Precision@K ---
fig, ax = plt.subplots(figsize=(10, 5))
prec_vals = [s1, s2, s3]
bars = ax.bar(systems_order, prec_vals, color=colors,
              edgecolor='white', linewidth=1.5)
ax.axhline(SM_BASELINE_RATE, color='red', linestyle='--',
           label=f'Baseline ({SM_BASELINE_RATE:.3f} = {SM_BASELINE_RATE*100:.1f}%)')
for bar, val in zip(bars, prec_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title(
    f'Stuffmart Validation — Precision@K (K=20% of contactable, n={SM_K_CONTACTABLE:,})\n'
    'S1=ML baseline | S2=Hard governance | S3=Calibrated robustness',
    fontsize=11, fontweight='bold'
)
ax.set_ylabel('Precision@K')
ax.set_ylim(0, max(prec_vals) * 1.3)
ax.legend()
plt.tight_layout()
plt.savefig('fig_SM_01_precision_comparison.png', dpi=150)
plt.show()

# --- SM-10b: Lift@K ---
fig, ax = plt.subplots(figsize=(10, 5))
lift_vals = [l1, l2, l3]
bars = ax.bar(systems_order, lift_vals, color=colors,
              edgecolor='white', linewidth=1.5)
ax.axhline(1.0, color='red', linestyle='--', label='Random selection (1.0x)')
for bar, val in zip(bars, lift_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.2f}x', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Stuffmart Validation — Lift@K (K=20% of contactable)',
             fontsize=11, fontweight='bold')
ax.set_ylabel('Lift@K')
ax.set_ylim(0, max(lift_vals) * 1.3)
ax.legend()
plt.tight_layout()
plt.savefig('fig_SM_02_lift_comparison.png', dpi=150)
plt.show()

# --- SM-10c: K sensitivity ---
fig, ax = plt.subplots(figsize=(11, 5))
for sys_name, color in zip(systems_order, colors):
    subset = sm_k_results[sm_k_results['system'] == sys_name]
    ax.plot(subset['k_fraction'] * 100, subset['precision_at_k'],
            marker='o', label=sys_name, color=color, linewidth=2)
ax.axhline(SM_BASELINE_RATE, color='red', linestyle='--',
           label=f'Baseline ({SM_BASELINE_RATE:.3f})', alpha=0.7)
ax.set_xlabel('K (% of contactable pool)', fontsize=10)
ax.set_ylabel('Precision@K', fontsize=10)
ax.set_title('Stuffmart — K Sensitivity (K = 10/20/30% of contactable pool)',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=8)
ax.set_xticks([10, 20, 30])
plt.tight_layout()
plt.savefig('fig_SM_03_k_sensitivity.png', dpi=150)
plt.show()

print("NOTE: H1 removes 50.2% of Stuffmart leads.")
print(f"Contactable pool: {len(sm_results[~sm_results['h1_flag']]):,} leads")
print(f"K at 20%: {SM_K_CONTACTABLE:,} leads")

IndexError: index 0 is out of bounds for axis 0 with size 0

In [ ]:
# =============================================================================
# CELL SM-11 — INTERPRETATION
# =============================================================================

print("=== STUFFMART SECONDARY VALIDATION INTERPRETATION ===\n")

r = sm_main_results.set_index("system")

s1 = r.loc["System 1 — ML baseline"]
s2 = r.loc["System 2 — Tier qualification"]
s3 = r.loc["System 3 — Calibrated blend"]

def pct(x):
    return f"{x * 100:.2f}%"

def pp(a, b):
    return f"{(a - b) * 100:+.2f} pp"

print("Main results:")
print(f"  S1 ML baseline:          Precision@K={pct(s1['precision_at_k'])}, Lift={s1['lift_at_k']:.2f}x")
print(f"  S2 Tier qualification:   Precision@K={pct(s2['precision_at_k'])}, Lift={s2['lift_at_k']:.2f}x")
print(f"  S3 Calibrated blend:     Precision@K={pct(s3['precision_at_k'])}, Lift={s3['lift_at_k']:.2f}x")

print("\nDirectional comparison:")
print(f"  S2 vs S1: {pp(s2['precision_at_k'], s1['precision_at_k'])}")
print(f"  S3 vs S1: {pp(s3['precision_at_k'], s1['precision_at_k'])}")

if s2["precision_at_k"] > s1["precision_at_k"]:
    direction = "positive"
elif np.isclose(s2["precision_at_k"], s1["precision_at_k"]):
    direction = "neutral"
else:
    direction = "negative"

print("\nInterpretation:")
print(f"  The Stuffmart validation produces a {direction} directional result for")
print("  the tier-based qualification mechanism relative to the ML baseline.")

print("""
Scope of conclusion:
  This is NOT a replication of the primary study. Stuffmart is synthetic,
  retail-oriented, highly imbalanced, and uses proxy signals that are only
  operationally analogous to the primary study's qualification variables.

  The validation answers a narrower question:
  does the same two-layer architecture — withholding selected signals from
  the model and applying them through a qualification layer — show directionally
  similar behaviour in a structurally different dataset?

  A positive result provides weak support that the mechanism is not purely an
  artefact of the X Education dataset. A neutral or negative result would show
  that the effect is context-dependent. In either case, the result should be
  reported as directional robustness evidence, not confirmatory external validity.
""")

In [ ]:

# =============================================================================
# CELL SM-12 — EXPORT ALL TABLES
# =============================================================================

sm_model_table = pd.DataFrame([
    {'model': 'Logistic Regression',           'auc_roc': round(sm_lr_auc,  6)},
    {'model': 'XGBoost',                       'auc_roc': round(sm_xgb_auc, 6)},
    {'model': f'Selected: {sm_best_model_name}', 'auc_roc': round(sm_best_auc, 6)},
])

print("=== TABLE SM-1: Model Benchmark ===")
print(sm_model_table.to_string(index=False))
sm_model_table.to_csv('table_SM_model_benchmark.csv', index=False)

print("\n=== TABLE SM-2: Main System Comparison (K=20%) ===")
print(sm_main_results.to_string(index=False))

print("\n=== TABLE SM-3: Tier Cap Sensitivity ===")
print(sm_cap_pivot.to_string())

print("\n=== TABLE SM-4: K Sensitivity ===")
print(sm_k_pivot.to_string())

print("\nAll Stuffmart tables exported.")
print("\nFigures saved:")
for f in [
    'fig_SM_01_precision_comparison.png',
    'fig_SM_02_lift_comparison.png',
    'fig_SM_03_cap_sensitivity.png',
    'fig_SM_04_cross_dataset_comparison.png',
]:
    print(f'  {f}')
